# Dynamical Systems Reconstruction with PLRNNs

This notebook demonstrates the full **Paradigm B** workflow in NeuralRNN: learning a generative RNN whose autonomous trajectories reproduce an observed dynamical system. We cover two complementary scenarios:

- **Part I — Lorenz-63**: reconstruct a low-dimensional chaotic attractor with an autonomous `shallowPLRNN`. We use analytic fixed points and standard DSR metrics (`D_stsp`, `D_H`, `λ_max`).
- **Part II — Task-state neural activity**: train a 50-unit CTRNN on the NeuroGym `PerceptualDecisionMaking-v0` task, collect 1000 trials of hidden-state activity together with the task inputs, and train a **non-autonomous** `shallowPLRNN` to reconstruct those task-driven dynamics. We then analyze the reconstructed model with PCA, vector fields, numeric fixed points, and Jacobian eigenvalues.

The only difference between the two parts is the **Objective**: `TeacherForcingObjective` for reconstruction versus `SupervisedObjective` for task training.

> Requirements: `pip install -e ..` (core deps). For NeuroGym tasks: `pip install -e '..[neurogym]'`.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

sys.path.append('../src')
from neuralrnn import (
    AutoConfig, AutoModel, Trainer, TrainingArguments, load_dataset,
    SupervisedObjective, TeacherForcingObjective,
)
from neuralrnn.data import TrialTimeseriesDataset
from neuralrnn.analysis import (
    fit_pca, find_fixed_points, linearize, dominant_direction,
    compute_vector_field, state_space_divergence, power_spectrum_error,
    max_lyapunov_exponent, compute_manifold,
)

torch.manual_seed(42)
np.random.seed(42)

# device = 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

SAVE_DIR = Path('./models/05')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Part I: Lorenz-63 — autonomous chaotic attractor reconstruction

We start with the classic Lorenz-63 system. The goal is to train a shallowPLRNN so that its free-run trajectories visit the same butterfly-shaped attractor as the true system.

### Model

For an autonomous system we set `input_dim=0`:

$$
z_t = A z_{t-1} + W_1 \mathrm{ReLU}(W_2 z_{t-1} + h_2) + h_1,
\qquad A=\mathrm{diag}.
$$

We use identity readout, so `latent_dim = output_dim = observation_dim = 3`. Generalized Teacher Forcing with a small `alpha` (0.1) stabilizes training on chaotic data.

In [ ]:
# Load the Lorenz-63 dataset
ds_lorenz = load_dataset('lorenz63', sequence_length=100, batch_size=128, normalize=True)
print('Observation dim N =', ds_lorenz.N, '| train length T =', ds_lorenz.T)

batch = ds_lorenz.sample_batch()
fig = plt.figure(figsize=(6, 3))
ax1 = fig.add_subplot(121, projection='3d')
for k, v in batch.items():
    if v is not None:
        print(k, tuple(v.shape))
        for i in range(v.shape[0]):
            ax1.plot3D(v[i, :, 0], v[i, :, 1], v[i, :, 2], lw=0.4, alpha=0.7, color='C0')
ax2 = fig.add_subplot(122, projection='3d')
v = ds_lorenz.test
ax2.plot3D(v[:, 0], v[:, 1], v[:, 2], lw=0.4, alpha=0.7, color='C1')    

### Build and train the autonomous shallowPLRNN

In [ ]:
lorenz_ckpt = SAVE_DIR / 'ckpt_lorenz_plrnn'
overwrite = False
if (lorenz_ckpt / 'config.json').exists() and not overwrite:
    print('Loading pre-trained Lorenz-63 PLRNN...')
    model_lorenz = AutoModel.from_pretrained(lorenz_ckpt)
else:
    cfg_lorenz = AutoConfig.for_model(
        'shallow_plrnn',
        input_dim=0,
        latent_dim=ds_lorenz.N,
        output_dim=ds_lorenz.N,
        hidden_dim=50,
        autonomous=True,
    )
    model_lorenz = AutoModel.from_config(cfg_lorenz)
    print(model_lorenz.__class__.__name__, '| #params =', model_lorenz.num_parameters())

    objective = TeacherForcingObjective(alpha=0.125)
    args = TrainingArguments(
        max_steps=8000,
        learning_rate=1e-3,
        grad_clip_norm=5.0,
        log_every=500,
    )
    Trainer(model_lorenz, ds_lorenz, objective, args).train()
    model_lorenz.save_pretrained(lorenz_ckpt)
    print('Saved to', lorenz_ckpt)

### Free-run evaluation and DSR metrics

We roll the trained model out without teacher forcing and compare the generated trajectory to the true test trajectory. `D_stsp` measures state-space distributional mismatch, `D_H` compares power spectra, and `λ_max` estimates the largest Lyapunov exponent.

**Important:** The shallowPLRNN is a discrete-time map trained on Lorenz-63 data sampled at `Δt = 0.01`. The raw QR-based exponent is therefore a *discrete-time* value. To compare with the continuous-time Lorenz-63 ground truth (`λ_max ≈ 0.906`), we divide the discrete-time estimate by `dt = 0.01`. 

In [ ]:
true = ds_lorenz.test.numpy() if ds_lorenz.test is not None else ds_lorenz.X.numpy()
z0 = torch.zeros(1, ds_lorenz.N)
gen = model_lorenz.generate(z0, n_steps=true.shape[0] - 1)[0].numpy()  # (T, N)

D_stsp = state_space_divergence(gen, true)
D_H = power_spectrum_error(true, gen, smoothing=20.0)
# The PLRNN is a discrete-time map trained on data sampled at dt=0.01.
# To compare with the continuous-time Lorenz-63 value (~0.906), divide by dt.
lam = max_lyapunov_exponent(
    model_lorenz,
    torch.tensor(gen[-1], dtype=torch.float32),
    T=10000,
    T_trans=1000,
    ons=10,
    dt=getattr(ds_lorenz, 'dt', 0.01),
)

print(f'D_stsp     = {D_stsp:.4f}')
print(f'D_H        = {D_H:.4f}')
print(f'lambda_max = {lam:.3f}')

In [ ]:
fig = plt.figure(figsize=(6, 3))
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(*true[:5000].T, lw=0.4, alpha=0.7)
ax1.set_title('True Lorenz-63')

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(*gen[:5000].T, lw=0.4, alpha=0.7)
ax2.set_title('Reconstructed')

plt.tight_layout()
plt.show()

### Analytic fixed points and stability

Because shallowPLRNN is piecewise-linear, its fixed points can be found analytically by enumerating ReLU configurations. We use `backend='auto'` (which selects the analytic solver for PLRNN-family models) and inspect the stability of each fixed point from the Jacobian eigenvalues.

In [ ]:
fps_lorenz = find_fixed_points(
    model_lorenz,
    backend='auto',
    max_order=1,
    outer_it=30,
    inner_it=10,
)
print(f'Found {len(fps_lorenz)} fixed points')
for p in fps_lorenz:
    print('  z* =', np.round(p.z, 3),
          '| stable:', p.is_stable,
          '| max|eig| =', round(float(np.max(np.abs(p.eigenvalues))), 3))

true_FPs = np.array([[-0.97025859,  1.16357464,  0.09665803], 
                      [-0.86533984,  1.03770608,  0.08618312],
                      [ 0.37320716,  0.37320716, -2.92500668]]).T


fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(*true[:2000].T, lw=0.4, alpha=0.7)
for p in fps_lorenz:
    ax1.scatter(*p.z, c='red' if not p.is_stable else 'blue', s=40)
for p in true_FPs:
    ax1.scatter(*p, c='green', s=40)
ax1.set_title('True attractor + fixed points')
ax1.legend(handles=[
    Line2D([0], [0], marker='o', color='w', label='estimated unstable FP', markerfacecolor='red', markersize=8),
    Line2D([0], [0], marker='o', color='b', label='estimated stable FP', markerfacecolor='blue', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='True FP', markerfacecolor='green', markersize=8),
])

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(*gen[:2000].T, lw=0.4, alpha=0.7)
for p in fps_lorenz:
    ax2.scatter(*p.z, c='red' if not p.is_stable else 'blue', s=40)
for p in true_FPs:
    ax2.scatter(*p, c='green', s=40)
ax2.set_title('Reconstruction + fixed points')

plt.tight_layout()
plt.show()

### 1.6 PLRNN-family variants on Lorenz-63

The same analytic toolbox works for all PLRNN-family models because they share a piecewise-linear structure. Here we repeat the Lorenz-63 reconstruction with `dend_plrnn` and `alrnn` and compare the results to the shallowPLRNN baseline.

- **dendPLRNN**: each latent unit has a linear spline basis expansion. We keep `latent_dim=3` so the readout remains identity and the trajectory is directly comparable to the true 3-D attractor.
- **ALRNN**: only the last `P = latent_dim - n_linear` latent units are ReLU; the rest are linear. With `latent_dim=3` and `n_linear=2` there is only `P=1` ReLU unit, which is too restrictive, so we use `n_linear=1` (`P=2`).

> Note: the variants are more sensitive to hyperparameters than shallowPLRNN, and Lorenz-63 is a challenging chaotic target. The numbers below illustrate the workflow; if `D_H` is `nan` or `lambda_max/dt` is far from 0.9, the model has likely collapsed to a non-chaotic attractor. You can tune `n_bases`, `n_linear`, `clip_range`, `alpha`, and training length for better metrics.

In [ ]:
def train_lorenz_variant(model_type, config_kwargs, alpha=0.125, max_steps=20000):
    """Train or load a PLRNN-family variant on the Lorenz-63 dataset."""
    ckpt = SAVE_DIR / f'ckpt_lorenz_{model_type}'
    if (ckpt / 'config.json').exists():
        print(f'Loading existing {model_type} checkpoint...')
        return AutoModel.from_pretrained(ckpt)
    cfg = AutoConfig.for_model(
        model_type,
        input_dim=0,
        latent_dim=ds_lorenz.N,
        output_dim=ds_lorenz.N,
        autonomous=True,
        **config_kwargs,
    )
    model = AutoModel.from_config(cfg)
    print(model.__class__.__name__, '| #params =', model.num_parameters())
    obj = TeacherForcingObjective(alpha=alpha)
    args = TrainingArguments(
        max_steps=max_steps,
        learning_rate=1e-3,
        grad_clip_norm=5.0,
        log_every=1000,
    )
    Trainer(model, ds_lorenz, obj, args).train()
    model.save_pretrained(ckpt)
    print('Saved', model_type, 'to', ckpt)
    return model

model_lorenz_dend = train_lorenz_variant(
    'dend_plrnn', {'n_bases': 20, 'clip_range': 5.0}, alpha=0.125, max_steps=10000
)
model_lorenz_alrnn = train_lorenz_variant(
    'alrnn', {'n_linear': 2, 'clip_range': 5.0}, alpha=0.125, max_steps=10000
)

#### Free-run evaluation and comparison

We roll out each variant from the first test point and report the same DSR metrics used for shallowPLRNN. Because all models use `latent_dim=3` and identity readout, the generated trajectories are directly comparable to the ground truth.

In [ ]:
def eval_lorenz_variant(model, name):
    """Evaluate a trained model on the Lorenz-63 test attractor."""
    z0 = torch.from_numpy(true[:1]).float()
    gen = model.generate(z0, n_steps=true.shape[0] - 1)[0].numpy()
    d_stsp = state_space_divergence(gen, true)
    d_h = power_spectrum_error(true, gen, smoothing=20.0)
    lam = max_lyapunov_exponent(
        model,
        torch.tensor(gen[-1], dtype=torch.float32),
        T=20000,
        T_trans=2000,
        ons=10,
        dt=getattr(ds_lorenz, 'dt', 0.01),
    )
    fps = find_fixed_points(
        model, backend='auto', max_order=1, outer_it=30, inner_it=10
    )
    d_h_str = f'{d_h:.4f}' if not np.isnan(d_h) else 'nan'
    print(f'{name:18s} D_stsp={d_stsp:.4f}  D_H={d_h_str}  '          f'lambda_max/dt={lam:.3f}  #FP={len(fps)}')
    return gen, fps

gen_lorenz_dend, fps_lorenz_dend = eval_lorenz_variant(model_lorenz_dend, 'dendPLRNN')
gen_lorenz_alrnn, fps_lorenz_alrnn = eval_lorenz_variant(model_lorenz_alrnn, 'ALRNN')

#### Side-by-side trajectory plots

The 3-D free-run trajectories let us visually compare how well each variant captures the butterfly-shaped Lorenz attractor.

In [ ]:
fig = plt.figure(figsize=(16, 4))
titles = ['True', 'shallowPLRNN', 'dendPLRNN', 'ALRNN']
trajs = [true[:5000], gen[:5000], gen_lorenz_dend[:5000], gen_lorenz_alrnn[:5000]]
for i, (traj, title) in enumerate(zip(trajs, titles)):
    ax = fig.add_subplot(1, 4, i + 1, projection='3d')
    ax.plot(*traj.T, lw=0.4, alpha=0.7)
    ax.set_title(title)
plt.tight_layout()
plt.show()

#### Variant fixed points

Because all three models expose effective `(A, W1, W2, h1, h2)` parameters, the analytic SCYFI solver finds their fixed points in the same way.

In [ ]:
for name, fps in [('dendPLRNN', fps_lorenz_dend), ('ALRNN', fps_lorenz_alrnn)]:
    print(f'{name}: {len(fps)} fixed points')
    for p in fps:
        print('  z* =', np.round(p.z, 3),
              '| stable:', p.is_stable,
              '| max|eig| =', round(float(np.max(np.abs(p.eigenvalues))), 3))

### Save / load round-trip

`save_pretrained` writes `config.json` + `model.safetensors`. Reloading should give bit-identical free-run outputs.

In [ ]:
reload_lorenz = AutoModel.from_pretrained(lorenz_ckpt)
reload_gen = reload_lorenz.generate(z0, n_steps=true.shape[0] - 1)[0].numpy()
print('Reload outputs match:', np.allclose(gen, reload_gen, atol=1e-6))

## Part II: Reconstructing task-state neural activity

We now move from autonomous systems to **task-driven neural dynamics**. The pipeline is:

1. Train a 50-dimensional CTRNN on the perceptual decision task (Paradigm A).
2. Collect 1000 trials of CTRNN hidden states and the task inputs that drove them.
3. Train a non-autonomous shallowPLRNN whose input is the task covariate and whose target is the hidden-state trajectory.
4. Analyze the reconstructed PLRNN with the same tools used in Part I, plus condition-wise PCA.

### 2.1 Train a CTRNN on `PerceptualDecisionMaking-v0`

The task presents a fixation cue plus left/right motion stimuli. The network must accumulate evidence and report the stimulus direction.

In [ ]:
ctrnn_ckpt = SAVE_DIR / 'ctrnn_task'

if (ctrnn_ckpt / 'config.json').exists():
    print('Loading pre-trained CTRNN...')
    ctrnn = AutoModel.from_pretrained(ctrnn_ckpt).to(device)
    ds_task = load_dataset('perceptual_decision_making', batch_size=16, seq_len=100, dt=100)
else:
    ds_task = load_dataset('perceptual_decision_making', batch_size=16, seq_len=100, dt=100)
    print('Task input_dim =', ds_task.input_dim, '| output_dim =', ds_task.output_dim)

    cfg_ctrnn = AutoConfig.for_model(
        'ctrnn',
        input_dim=ds_task.input_dim,
        latent_dim=50,
        output_dim=ds_task.output_dim,
        dt=100,
        tau=100,
        nonlinearity_mode='pre_activation',
    )
    ctrnn = AutoModel.from_config(cfg_ctrnn).to(device)
    print(ctrnn.__class__.__name__, '| #params =', ctrnn.num_parameters())

    objective_ctrnn = SupervisedObjective(task_type='classification')
    args_ctrnn = TrainingArguments(
        max_steps=2000, learning_rate=1e-3, grad_clip_norm=1.0, log_every=200, device=str(device),
    )
    Trainer(ctrnn, ds_task, objective_ctrnn, args_ctrnn).train()
    ctrnn.save_pretrained(ctrnn_ckpt)
    print('Saved CTRNN to', ctrnn_ckpt)

### 2.2 Collect 1000 trials of hidden-state activity and task inputs

For each trial we run the trained CTRNN forward and store:

- `states`: the CTRNN hidden activity of shape `(T, 50)`. This becomes the **observation** for the PLRNN.
- `ob`: the task input of shape `(T, 3)` (fixation + left stimulus + right stimulus). This becomes the **external input** for the PLRNN.

We also record trial metadata (`ground_truth` and coherence) so we can color PCA plots by task condition.

In [ ]:
# Collect CTRNN hidden-state activity and task inputs over complete trials
# (trial-aligned dataset interface — same as the built-in cognitive tasks)
num_trial = 1000
# Complete trials straight from the training dataset (no new dataset needed)
ds_collect = ds_task.sample_trials(num_trial)

ctrnn.eval()
with torch.no_grad():
    out = ctrnn(ds_collect.inputs.to(device))   # batched forward over all trials
states_all = out.states.cpu().numpy()           # (N, T, 50)

acts, inputs, meta = [], [], []
for i, cond in enumerate(ds_collect.conditions):
    n = cond['n_steps']                         # true length of this trial (unpadded)
    acts.append(states_all[i, :n])              # (T, 50)
    inputs.append(ds_collect.inputs[i, :n].numpy())  # (T, 3)

    coh = cond.get('coh', 0)
    if hasattr(coh, '__iter__'):
        coh = float(np.mean(list(coh)))
    meta.append({
        'trial': i,
        'length': n,
        'ground_truth': int(cond['ground_truth']),
        'coh': float(coh),
    })

# Preserve trial structure: (n_trial, trial_length, n_variable)
activity_trials = np.stack(acts, axis=0)      # (1000, T, 50)
input_trials = np.stack(inputs, axis=0)       # (1000, T, 3)
meta_df = pd.DataFrame(meta)
print('Collected activity trials:', activity_trials.shape)
print('Collected input trials:  ', input_trials.shape)
print('Trials:', len(meta_df))
print(meta_df.head())

### 2.3 Build a non-autonomous shallowPLRNN dataset

For Paradigm B the observed neural trajectory goes into `inputs`/`targets`, and the task covariate goes into `external_inputs`. Because each trial is an independent sequence, we use `TrialTimeseriesDataset`, which preserves the `(n_trial, trial_length, n_variable)` structure and performs trial-wise train/test splitting. This avoids leakage across trial boundaries that would occur if we concatenated all trials and used sliding windows.

We normalize the neural activity (recommended because CTRNN firing rates and task inputs live on different scales). We keep the raw task inputs by setting `normalize_externals=False`; you can set it to `True` if you want both signals on comparable scales.

In [ ]:
ds_plrnn = TrialTimeseriesDataset.from_arrays(
    activity_trials,
    external_inputs=input_trials,
    batch_size=64,
    normalize=True,
    normalize_externals=False,
    test_fraction=0.1,
    seed=0,
)

batch = ds_plrnn.sample_batch()
for k, v in batch.items():
    if v is not None:
        print(k, tuple(v.shape))

print('External-input normalizer present:', ds_plrnn.external_normalizer is not None)
print('Train trials:', len(ds_plrnn), '| Test trials:', len(ds_plrnn.test_set) if ds_plrnn.test_set is not None else 0)

### 2.4 Train the non-autonomous shallowPLRNN

We set `input_dim=3` (task input), `latent_dim=50` (must equal the observation dimension so `TeacherForcingObjective` forces the full latent state), and `autonomous=False`.

In [ ]:
plrnn_ckpt = SAVE_DIR / 'plrnn_task'
overwrite = False
if (plrnn_ckpt / 'config.json').exists() and not overwrite:
    print('Loading pre-trained task-state PLRNN...')
    model_plrnn = AutoModel.from_pretrained(plrnn_ckpt).to(device)
else:
    cfg_plrnn = AutoConfig.for_model(
        'shallow_plrnn',
        input_dim=input_trials.shape[-1],         # 3 (external task input dim)
        latent_dim=activity_trials.shape[-1],     # 50
        output_dim=activity_trials.shape[-1],     # 50
        hidden_dim=200,
        autonomous=False,
    )
    model_plrnn = AutoModel.from_config(cfg_plrnn).to(device)
    print(model_plrnn.__class__.__name__, '| #params =', model_plrnn.num_parameters())

    objective_plrnn = TeacherForcingObjective(alpha=0.1)
    args_plrnn = TrainingArguments(
        max_steps=8000,
        learning_rate=1e-3,
        grad_clip_norm=10.0,
        log_every=200,
        device=str(device),
    )
    Trainer(model_plrnn, ds_plrnn, objective_plrnn, args_plrnn).train()
    model_plrnn.save_pretrained(plrnn_ckpt)
    print('Saved PLRNN to', plrnn_ckpt)

### 2.5 Reconstruction quality on held-out task inputs

Because the PLRNN is non-autonomous, a fair free-run evaluation must feed it the same task inputs that were presented during the test set. We use `model.generate(..., inputs=...)` and compare the generated trajectory to the true normalized activity.

In [ ]:
# Free-run evaluation on held-out trials. The test_set contains whole trials.
test_ds = ds_plrnn.test_set
test_activity = test_ds.X.reshape(-1, test_ds.input_dim).numpy()          # (T_test, 50), normalized
test_inputs = test_ds.S.reshape(-1, test_ds.S.shape[-1]).numpy()          # (T_test, 3)

z0 = torch.zeros(1, model_plrnn.config.latent_dim, device=device)
inputs_tensor = torch.from_numpy(test_inputs).float().unsqueeze(0).to(device)  # (1, T, 3)
gen_task = model_plrnn.generate(z0, n_steps=test_activity.shape[0] - 1, inputs=inputs_tensor)[0].cpu().numpy()

D_stsp_task = state_space_divergence(gen_task, test_activity)
D_H_task = power_spectrum_error(test_activity, gen_task, smoothing=20.0)
if isinstance(D_H_task, np.ndarray):
    D_H_mean = float(D_H_task.mean())
    D_H_std = float(D_H_task.std())
else:
    D_H_mean = float(D_H_task)
    D_H_std = 0.0

print(f'Test D_stsp = {D_stsp_task:.4f}')
print(f'Test D_H    = {D_H_mean:.4f} ± {D_H_std:.4f} (mean ± std over channels)')

In [ ]:
# Visualize a few example channels
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
axes = axes.flatten()
t = np.arange(test_activity.shape[0])
for i, ax in enumerate(axes):
    ax.plot(t, test_activity[:, i], label='True', alpha=0.7)
    ax.plot(t, gen_task[:, i], label='Reconstructed', alpha=0.7)
    ax.set_title(f'Channel {i}')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Normalized activity')
axes[0].legend()
plt.tight_layout()
plt.show()

### 2.6 Per-condition PCA

We fit PCA on the training activity and project test trials. Trials are colored by the animal's correct choice (`ground_truth`). This reveals whether the reconstructed PLRNN organises its state space according to task conditions.

In [ ]:
# Fit PCA on the full normalized activity (flattened across trials) so trial-level labels align.
activity_for_pca = ds_plrnn.normalizer.transform(
    torch.from_numpy(activity_trials.reshape(-1, activity_trials.shape[-1])).float()
).numpy()

pca = fit_pca(activity_for_pca, n_components=3)
print('PCA explained variance:', np.round(pca.explained_variance_ratio, 3))
print('Cumulative:', round(float(pca.explained_variance_ratio.sum()), 3))

# Build per-time-step condition labels by repeating trial-level metadata
trial_lengths = meta_df['length'].values
choice_labels = np.repeat(meta_df['ground_truth'].values, trial_lengths)
coh_labels = np.repeat(meta_df['coh'].values, trial_lengths)

# Project the full activity
activity_pc = pca.transform(activity_for_pca)
print('PCA projection shape:', activity_pc.shape)
print('Label shape:', choice_labels.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

colors_choice = {1: '#2196F3', 0: '#F44336'}
for choice in [0, 1]:
    mask = choice_labels == choice
    axes[0].scatter(activity_pc[mask, 0][:1000], activity_pc[mask, 1][:1000],
                    c=colors_choice[choice], s=1, alpha=0.2,
                    label=f'Choice {choice}')
axes[0].set_xlabel('PC 1')
axes[0].set_ylabel('PC 2')
axes[0].set_title('PCA colored by ground-truth choice')
axes[0].legend(markerscale=6)
axes[0].grid(True, alpha=0.2)

sc = axes[1].scatter(activity_pc[:, 0], activity_pc[:, 1],
                     c=coh_labels, cmap='Reds', s=1, alpha=0.3)
axes[1].set_xlabel('PC 1')
axes[1].set_ylabel('PC 2')
axes[1].set_title('PCA colored by coherence')
plt.colorbar(sc, ax=axes[1], label='Coherence')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

### 2.7 Vector field in the PCA plane

`compute_vector_field` projects the one-step PLRNN map onto the first two PCs. We evaluate it at the zero-coherence task input `[fixation=1, left=0.5, right=0.5]`, matching the convention used in `01_ctrnn_fixedpoints_paradigmA.ipynb`.

In [ ]:
task_input_zero = torch.tensor([1.0, 0.5, 0.5], dtype=torch.float32)

vf = compute_vector_field(
    model_plrnn,
    basis=pca.components[:2],
    mean=pca.mean,
    task_input=task_input_zero,
    extent=(-4.0, 6.0),
    n_grid=15,
)

fig, ax = plt.subplots(figsize=(7, 6))
# Overlay a subset of test trajectories
n_show = min(20, len(acts))
for i in range(n_show):
    traj_pc = pca.transform(acts[i])
    ax.plot(traj_pc[:, 0], traj_pc[:, 1], color='gray', alpha=0.8, lw=1)

X = vf.grid_pc[:, :, 0]
Y = vf.grid_pc[:, :, 1]
U = vf.velocity_pc[:, :, 0]
V = vf.velocity_pc[:, :, 1]
ax.quiver(X, Y, U, V, color='darkblue', scale=None, width=0.002, alpha=0.7)

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Vector field (zero-coherence input)')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 2.8 Fixed points under a task condition

For the non-autonomous PLRNN we can still use the **analytic** fixed-point backend. With a constant task input `s*`, the fixed-point equation is

\[
z^* = A z^* + W_1 \mathrm{ReLU}(W_2 z^* + h_2) + h_1 + C s^*.
\]

This is equivalent to replacing the bias with `h1_eff = h1 + C s*` and solving the autonomous equation. The framework folds the task input into the bias before invoking SCYFI, so we do not need to fall back to numeric search.

In [ ]:
fps_task = find_fixed_points(
    model_plrnn,
    backend='auto',
    task_input=task_input_zero,
    max_order=1,
    outer_it=30,
    inner_it=10,
)
print(f'Found {len(fps_task)} fixed points for zero-coherence input')
for p in fps_task:
    print('  z* =', np.round(p.z, 2),
          '| stable:', p.is_stable,
          '| max|eig| =', round(float(np.max(np.abs(p.eigenvalues))), 3))

### 2.9 Jacobian eigenvalues and dominant direction

The eigenvalues of the Jacobian at a fixed point tell us whether nearby trajectories converge (stable) or diverge (unstable). The dominant eigenvector points along the direction of fastest escape.

In [ ]:
if len(fps_task):
    best_idx = int(np.argmin([p.speed for p in fps_task]))
    fp_ref = fps_task.points[best_idx]

    lin = linearize(model_plrnn, fp_ref.z, task_input=task_input_zero)
    eigvals = lin.eigenvalues
    d = dominant_direction(lin)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Complex plane
    ax1.scatter(np.real(eigvals), np.imag(eigvals), s=30, alpha=0.7, c='steelblue',
                edgecolors='white', linewidths=0.5)
    theta = np.linspace(0, 2 * np.pi, 200)
    ax1.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, lw=0.8,
             label='|λ|=1')
    ax1.axvline(0, color='gray', ls='--', alpha=0.3)
    ax1.axhline(0, color='gray', ls='--', alpha=0.3)
    ax1.set_xlabel('Re(λ)')
    ax1.set_ylabel('Im(λ)')
    ax1.set_title('Jacobian eigenvalues at best fixed point')
    ax1.set_aspect('equal')
    ax1.legend()
    ax1.grid(True, alpha=0.2)

    # Dominant direction on PCA
    fp_pc = pca.transform(fp_ref.z.reshape(1, -1))[0]
    ends = np.stack([fp_ref.z + 2 * d, fp_ref.z - 2 * d])
    ends_pc = pca.transform(ends)

    ax2.scatter(activity_pc[:, 0], activity_pc[:, 1], s=1, alpha=0.1, c='gray')
    ax2.scatter(fp_pc[0], fp_pc[1], c='red', s=80, zorder=5, marker='x')
    ax2.plot(ends_pc[:, 0], ends_pc[:, 1], 'r-', lw=2, label='Dominant direction')
    ax2.set_xlabel('PC 1')
    ax2.set_ylabel('PC 2')
    ax2.set_title('Fixed point and dominant eigenvector in PC space')
    ax2.legend()
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()
else:
    print('No fixed points found; try increasing n_candidates or speed_tol.')

### 2.10 Stable/unstable manifold tracing (DetectingManifolds)

Fixed points alone only tell us where the state can rest; their stable and unstable manifolds tell us how the state space is organized. Following Eisenmann et al. (2025), we trace the local stable/unstable manifold of a fixed point across the piecewise-linear regions of the PLRNN.

For a fixed point under constant task input, the local manifold is spanned by the eigenvectors of the Jacobian with eigenvalues inside (stable) or outside (unstable) the unit circle. We sample points along these eigenvectors, propagate them forward or backward until they cross into a new ReLU region, fit a PCA segment in the new region, and repeat.

Below we trace the manifold of the dominant fixed point found under the zero-coherence input and project the segments onto the PCA plane. Because the task-state PLRNN lives in 50 dimensions, the full global manifold may be curved; the trace gives a local approximation near the fixed point. If the fixed point is stable, we trace its stable manifold; if it is a saddle, we trace the unstable manifold.

In [ ]:
from neuralrnn.analysis import compute_manifold

# Pick the fixed point with the largest eigenvalue magnitude to trace its manifold.
# Under the zero-coherence input the PLRNN may settle to a stable fixed point;
# we can still trace its stable or unstable directions depending on the spectrum.
fp_ref = max(fps_task.points, key=lambda p: np.max(np.abs(p.eigenvalues)))
print('Tracing manifold from fixed point with max|eig| =',
      round(float(np.max(np.abs(fp_ref.eigenvalues))), 3),
      '| stable:', fp_ref.is_stable)

# Trace the unstable direction if any eigenvalue is outside the unit circle,
# otherwise trace the stable direction.
trace_stable = bool(fp_ref.is_stable)
manifold = compute_manifold(
    model_plrnn,
    fixed_point=fp_ref.z,
    task_input=task_input_zero,
    stable=trace_stable,
    n_samples=300,
    factor=0.05,
    max_iter=3,
    propagation_steps=50,
)

print(f'Traced {len(manifold.segments)} {"stable" if trace_stable else "unstable"}-manifold segments')

# Project segments onto the PCA plane
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(activity_pc[:, 0], activity_pc[:, 1], s=1, alpha=0.1, c='gray')

for seg in manifold.segments:
    seg_pc = pca.transform(seg.points)
    ax.scatter(seg_pc[:, 0], seg_pc[:, 1], s=5, alpha=0.5, label=f'region {seg.region_id}')

# Mark the fixed point
fp_pc = pca.transform(fp_ref.z.reshape(1, -1))[0]
ax.scatter(fp_pc[0], fp_pc[1], c='red', s=100, marker='x', zorder=5, label='fixed point')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title(f'{"Stable" if trace_stable else "Unstable"} manifold segments under zero-coherence input')
ax.legend(markerscale=4)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 2.11 Task-state reconstruction with dendPLRNN and ALRNN

We now repeat the non-autonomous reconstruction using the same 1000-trial CTRNN hidden states and task inputs, but train `dend_plrnn` and `alrnn` instead of `shallow_plrnn`. This shows that the full PLRNN family can be used for task-driven neural dynamics.

- **dendPLRNN**: `latent_dim=50, n_bases=10`.
- **ALRNN**: `latent_dim=50, n_linear=40` (so `P=10` latent units are ReLU). ALRNN training typically benefits from sparse identity teacher forcing, so we use `alpha=0.5` with `forcing_interval=10`.

> Note: `clip_range=5.0` is added to both task-state variants to prevent latent states from exploding during free-run evaluation. If `D_H` reports `nan`, some reconstructed channels have near-zero variance; this can happen when a model collapses to a stable fixed point or limit cycle under the held-out inputs.

In [ ]:
def train_task_variant(model_type, config_kwargs, alpha, max_steps):
    """Train or load a PLRNN-family variant on the task-state dataset."""
    ckpt = SAVE_DIR / f'ckpt_task_{model_type}'
    if (ckpt / 'config.json').exists():
        print(f'Loading existing {model_type} checkpoint...')
        return AutoModel.from_pretrained(ckpt).to(device)
    cfg = AutoConfig.for_model(
        model_type,
        input_dim=input_trials.shape[-1],
        latent_dim=activity_trials.shape[-1],
        output_dim=activity_trials.shape[-1],
        autonomous=False,
        **config_kwargs,
    )
    model = AutoModel.from_config(cfg).to(device)
    print(model.__class__.__name__, '| #params =', model.num_parameters())
    if model_type == 'alrnn':
        obj = TeacherForcingObjective(alpha=alpha, forcing_interval=10)
    else:
        obj = TeacherForcingObjective(alpha=alpha)
    args = TrainingArguments(
        max_steps=max_steps,
        learning_rate=1e-3,
        grad_clip_norm=10.0,
        log_every=500,
        device=str(device),
    )
    Trainer(model, ds_plrnn, obj, args).train()
    model.save_pretrained(ckpt)
    print('Saved', model_type, 'to', ckpt)
    return model

model_task_dend = train_task_variant(
    'dend_plrnn', {'n_bases': 10, 'clip_range': 5.0}, alpha=0.1, max_steps=8000
)
model_task_alrnn = train_task_variant(
    'alrnn', {'n_linear': 40, 'clip_range': 5.0}, alpha=0.5, max_steps=12000
)

#### Held-out reconstruction quality

We feed the held-out test inputs to each variant and compare the generated trajectories to the true normalized CTRNN activity, using the same test set constructed for shallowPLRNN.

In [ ]:
def eval_task_variant(model, name):
    """Evaluate a task-state variant on the held-out test trials."""
    z0 = torch.zeros(1, model.config.latent_dim, device=device)
    inputs_t = torch.from_numpy(test_inputs).float().unsqueeze(0).to(device)
    gen_t = model.generate(
        z0, n_steps=test_activity.shape[0] - 1, inputs=inputs_t
    )[0].cpu().numpy()
    d_stsp = state_space_divergence(gen_t, test_activity)
    d_h = power_spectrum_error(test_activity, gen_t, smoothing=20.0)
    d_h_mean = float(d_h.mean()) if isinstance(d_h, np.ndarray) else float(d_h)
    d_h_str = f'{d_h_mean:.4f}' if not np.isnan(d_h_mean) else 'nan'
    print(f'{name:18s} Test D_stsp={d_stsp:.4f}  Test D_H={d_h_str}')
    return gen_t

gen_task_dend = eval_task_variant(model_task_dend, 'dendPLRNN')
gen_task_alrnn = eval_task_variant(model_task_alrnn, 'ALRNN')

#### Example channel traces

A few example channels from the held-out test set, comparing the true CTRNN activity to the three PLRNN reconstructions.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
axes = axes.flatten()
t = np.arange(test_activity.shape[0])
models_gen = [
    ('shallowPLRNN', gen_task),
    ('dendPLRNN', gen_task_dend),
    ('ALRNN', gen_task_alrnn),
]
for i, ax in enumerate(axes):
    ax.plot(t, test_activity[:, i], label='True', alpha=0.7, color='black')
    for label, g in models_gen:
        ax.plot(t, g[:, i], label=label, alpha=0.6)
    ax.set_title(f'Channel {i}')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Normalized activity')
axes[0].legend()
plt.tight_layout()
plt.show()

#### Per-condition PCA and vector field for variants

We project the variant-generated trajectories onto the PCA basis fit on the true training activity. This checks whether the reconstructed dynamics organize the state space in the same task-conditional way as the teacher CTRNN.

In [ ]:
# Generate full-dataset trajectories so the PCA masks (which cover all 1000 trials) line up.
# Here we concatenate trials into one long sequence; this is acceptable for a density
# visualization, though strictly speaking each trial is an independent initial condition.
all_inputs_t = torch.from_numpy(input_trials).float().reshape(1, -1, input_trials.shape[-1]).to(device)

z0_all = torch.zeros(1, model_task_dend.config.latent_dim, device=device)
gen_task_dend_all = model_task_dend.generate(
    z0_all, n_steps=all_inputs_t.shape[1] - 1, inputs=all_inputs_t
)[0].cpu().numpy()
gen_task_dend_all_pc = pca.transform(gen_task_dend_all)

z0_all = torch.zeros(1, model_task_alrnn.config.latent_dim, device=device)
gen_task_alrnn_all = model_task_alrnn.generate(
    z0_all, n_steps=all_inputs_t.shape[1] - 1, inputs=all_inputs_t
)[0].cpu().numpy()
gen_task_alrnn_all_pc = pca.transform(gen_task_alrnn_all)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, gpc) in zip(axes, [
    ('shallowPLRNN', activity_pc),
    ('dendPLRNN', gen_task_dend_all_pc),
    ('ALRNN', gen_task_alrnn_all_pc),
]):
    for choice in [0, 1]:
        mask = choice_labels == choice
        ax.scatter(gpc[mask, 0][:1000], gpc[mask, 1][:1000],
                   c=colors_choice[choice], s=1, alpha=0.2,
                   label=f'Choice {choice}')
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.set_title(label)
    ax.legend(markerscale=6)
    ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

#### Fixed points under zero-coherence input

Finally, we use the analytic backend to locate fixed points of each variant under the constant zero-coherence task input. The framework folds the input into the effective bias, so the same SCYFI solver applies.

In [ ]:
for name, model in [('dendPLRNN', model_task_dend), ('ALRNN', model_task_alrnn)]:
    fps = find_fixed_points(
        model,
        backend='auto',
        task_input=task_input_zero,
        max_order=1,
        outer_it=30,
        inner_it=10,
    )
    print(f'{name}: found {len(fps)} fixed points for zero-coherence input')
    for p in fps:
        print('  z* =', np.round(p.z, 2),
              '| stable:', p.is_stable,
              '| max|eig| =', round(float(np.max(np.abs(p.eigenvalues))), 3))

### 2.12 Save models and collected data

In [ ]:
np.savez(
    SAVE_DIR / 'task_state_data.npz',
    activity_train=ds_plrnn.X.numpy(),
    inputs_train=ds_plrnn.S.numpy(),
    activity_test=ds_plrnn.test_set.X.numpy(),
    inputs_test=ds_plrnn.test_set.S.numpy(),
    trial_meta=meta_df.to_records(index=False),
)
print('Saved collected data to', SAVE_DIR / 'task_state_data.npz')
print('Saved models under', SAVE_DIR)

## Summary

This notebook walked through the two main use cases of the NeuralRNN PLRNN family:

| Scenario | Model | Objective | Input | Fixed-point backend |
|---|---|---|---|---|
| Autonomous attractor (Lorenz-63) | `shallow_plrnn` autonomous | `TeacherForcingObjective(alpha=0.125)` | none | analytic |
| Autonomous attractor (Lorenz-63) | `dend_plrnn` autonomous | `TeacherForcingObjective(alpha=0.125)` | none | analytic |
| Autonomous attractor (Lorenz-63) | `alrnn` autonomous | `TeacherForcingObjective(alpha=0.5)` | none | analytic |
| Task-driven neural activity | `shallow_plrnn` non-autonomous | `TeacherForcingObjective(alpha=0.1)` | task inputs via `external_inputs` | **analytic** (task input folded into bias) |
| Task-driven neural activity | `dend_plrnn` non-autonomous | `TeacherForcingObjective(alpha=0.1)` | task inputs via `external_inputs` | **analytic** (task input folded into bias) |
| Task-driven neural activity | `alrnn` non-autonomous | `TeacherForcingObjective(alpha=0.5, forcing_interval=10)` | task inputs via `external_inputs` | **analytic** (task input folded into bias) |

Key take-aways:

1. **Paradigm B = right Objective**: the same `Trainer` and model base class handle both reconstruction and task training; only the objective changes.
2. **Non-autonomous PLRNNs** reconstruct task-conditioned neural activity by feeding task covariates through `external_inputs` and setting `autonomous=False`.
3. For task-state models, the **analytic** fixed-point backend works when the task input is held constant: the framework computes `h1_eff = h1 + C s*` and runs SCYFI on the equivalent autonomous map.
4. **Trial-aware data** are essential for task-state reconstruction: use `TrialTimeseriesDataset` to keep each trial as an independent sequence and avoid sliding windows that cross trial boundaries.
5. The Lorenz-63 `lambda_max` must be divided by the sampling interval `dt=0.01` to compare with the continuous-time ground truth (~0.906).

For the task-optimization perspective (training the CTRNN teacher), see `01_ctrnn_fixedpoints_paradigmA.ipynb`; for a focused comparison of the three PLRNN-family variants on Lorenz-63, see `02a_plrnn_variants_dend_alrnn.ipynb`.